# Google Cloud Storage Dataset Manager

This notebook demonstrates how to use the `Datasets` class for managing datasets in Google Cloud Storage (GCS). The module supports operations like uploading, reading, updating, deleting, and listing datasets in various formats (CSV, JSON, Pickle). 

The `Datasets` class provides the following methods:
1. `upload_dataset` – Uploads a dataset to GCS in the specified format.
2. `read_dataset` – Reads a dataset from GCS.
3. `update_dataset` – Updates an existing dataset on GCS.
4. `delete_dataset` – Deletes a dataset from GCS.
5. `list_datasets` – Lists all datasets in the specified bucket.

We will demonstrate these methods step by step.

---

## 1. Initialize the Datasets Class

Before performing any operations, we need to initialize the `Datasets` class by authenticating with Google Cloud and specifying the project ID and bucket name.


In [4]:
# Import necessary modules
import warnings
import json
import csv
import pickle
import os
import pandas as pd
from google.cloud import storage
import io
from typing import Union, List, Dict, Any, Optional

# Set the path to your service account key file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Users/mac/Downloads/strkfarm-88de68f85013.json"

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning, module='urllib3')

# Constants for project ID and bucket name
DEFAULT_PROJECT_ID = "strkfarm"
DEFAULT_BUCKET_NAME = "strkfarm"

# Datasets class with all necessary methods
class Datasets:
    def __init__(self, project_id: str = DEFAULT_PROJECT_ID, bucket_name: str = DEFAULT_BUCKET_NAME) -> None:
        self.project_id = project_id
        self.bucket_name = bucket_name

        try:
            self.storage_client = storage.Client(project=self.project_id)
            self.bucket = self.storage_client.bucket(self.bucket_name)
            self.bucket.reload()  # Verify the bucket exists and is accessible
            print("Successfully authenticated to Google Cloud Storage.")
        except Exception as e:
            print(f"Error authenticating or accessing bucket: {e}")
            raise

    SUPPORTED_FORMATS = ["json", "csv", "pickle"]

    def upload_dataset(self, 
                       data: Union[pd.DataFrame, Dict, List, Any], 
                       filename: str, 
                       data_format: str = "json") -> None:
        """
        Uploads data to Google Cloud Storage in the exact specified format.

        Args:
            data: The data to upload. Can be a pandas DataFrame, dict, list, or other serializable object.
            filename: The target filename in GCS.
            data_format: The format to store the data ("json", "csv", or "pickle"). Defaults to "json".

        Raises:
            ValueError: If the data_format is not supported.
            Exception: If upload fails.
        """
        if data_format not in self.SUPPORTED_FORMATS:
            raise ValueError(f"Unsupported data format: {data_format}. Must be one of {self.SUPPORTED_FORMATS}")

        try:
            # Handle different input data types and formats
            if isinstance(data, pd.DataFrame):
                blob_data = data.to_csv(index=False).encode('utf-8')
                content_type = 'text/csv'
            elif data_format == "json":
                blob_data = json.dumps(data).encode('utf-8')
                content_type = 'application/json'
            elif data_format == "pickle":
                blob_data = pickle.dumps(data)
                content_type = 'application/octet-stream'

            # Upload to GCS
            blob = self.bucket.blob(filename)
            blob.upload_from_string(blob_data, content_type=content_type)
            print(f"Dataset '{filename}' uploaded successfully.")

        except Exception as e:
            print(f"Error uploading dataset: {e}")
            raise

    def read_dataset(self, 
                     filename: str, 
                     data_format: str = "json",
                     as_dataframe: bool = False) -> Optional[Union[pd.DataFrame, Dict, List, Any]]:
        """
        Reads data from Google Cloud Storage in the specified format.

        Args:
            filename: The name of the file to read from GCS.
            data_format: The format of the stored data ("json", "csv", or "pickle"). Defaults to "json".
            as_dataframe: If True and format is "csv", returns a pandas DataFrame. Defaults to False.

        Returns:
            The loaded data in its appropriate Python format, or None if reading fails.

        Raises:
            ValueError: If the data_format is not supported.
            Exception: If read operation fails.
        """
        if data_format not in self.SUPPORTED_FORMATS:
            raise ValueError(f"Unsupported data format: {data_format}. Must be one of {self.SUPPORTED_FORMATS}")

        try:
            blob = self.bucket.blob(filename)
            blob_data = blob.download_as_bytes()

            if data_format == "json":
                return json.loads(blob_data.decode('utf-8'))
            elif data_format == "csv":
                csv_data = self._read_csv(blob_data)
                if as_dataframe:
                    return pd.DataFrame(csv_data[1:], columns=csv_data[0])
                return csv_data
            elif data_format == "pickle":
                return pickle.loads(blob_data)

        except Exception as e:
            print(f"Error reading {data_format} data: {e}")
            return None

    def _read_csv(self, data: bytes) -> List[List[str]]:
        """
        Helper function to read CSV data.

        Args:
            data: The CSV data as bytes.

        Returns:
            List[List[str]]: List of rows from the CSV data.

        Raises:
            Exception: If CSV parsing fails.
        """
        try:
            reader = csv.reader(io.StringIO(data.decode('utf-8')))
            return list(reader)
        except Exception as e:
            print(f"Error parsing CSV: {e}")
            raise

    def update_dataset(self, 
                       data: Union[pd.DataFrame, Dict, List, Any], 
                       filename: str, 
                       data_format: str = "json") -> None:
        """
        Updates (overwrites) an existing dataset in Google Cloud Storage.

        This is a wrapper around upload_dataset that makes the update operation explicit.
        Args:
            data: The new data to upload.
            filename: The name of the file to update.
            data_format: The format to store the data ("json", "csv", or "pickle"). Defaults to "json".

        Raises:
            ValueError: If the data_format is not supported.
            Exception: If update fails.
        """
        self.upload_dataset(data, filename, data_format)

    def delete_dataset(self, filename: str) -> bool:
        """
        Deletes a dataset from Google Cloud Storage.

        Args:
            filename: The name of the file to delete.

        Returns:
            bool: True if deletion was successful, False otherwise.

        Raises:
            Exception: If deletion fails.
        """
        blob = self.bucket.blob(filename)
        try:
            blob.delete()
            print(f"Dataset '{filename}' deleted successfully.")
            return True
        except Exception as e:
            print(f"Error deleting dataset: {e}")
            return False

    def list_datasets(self, prefix: str = None) -> List[str]:
        """
        Lists all datasets in the bucket, optionally filtered by prefix.

        Args:
            prefix: Optional prefix to filter the files. Defaults to None.

        Returns:
            List[str]: List of dataset filenames.
        """
        try:
            blobs = self.bucket.list_blobs(prefix=prefix)
            return [blob.name for blob in blobs]
        except Exception as e:
            print(f"Error listing datasets: {e}")
            return []

# Example usage
datasets = Datasets()


Successfully authenticated to Google Cloud Storage.


Now that the `Datasets` class is initialized, we can start demonstrating how to interact with Google Cloud Storage.

---

## 2. Upload a Dataset to GCS

The `upload_dataset` method uploads a dataset in one of the supported formats (`csv`, `json`, `pickle`). We'll demonstrate uploading a pandas DataFrame as a CSV file.


In [5]:
# Create an instance of Datasets class
datasets = Datasets()

# Example data to upload
data = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "age": [23, 30, 35]
})

# Upload data as CSV
filename = "people_data.csv"
datasets.upload_dataset(data, filename, data_format="csv")


Successfully authenticated to Google Cloud Storage.
Dataset 'people_data.csv' uploaded successfully.


In this example, we've uploaded a simple DataFrame as a CSV file to Google Cloud Storage. The dataset is now available in the GCS bucket under the name `people_data.csv`.

---

## 3. Read a Dataset from GCS

Next, we demonstrate how to read a dataset from Google Cloud Storage. We'll read the CSV file we just uploaded and load it into a pandas DataFrame.


In [1]:
# Read the uploaded CSV dataset from GCS
read_data = datasets.read_dataset(filename, data_format="csv", as_dataframe=True)

# Display the loaded data
read_data


NameError: name 'datasets' is not defined

The `read_dataset` method successfully loads the dataset from GCS and returns it as a pandas DataFrame.

---

## 4. Update an Existing Dataset

If we need to update an existing dataset, we can use the `update_dataset` method, which works as a wrapper around the `upload_dataset` method to overwrite the existing dataset.

For this example, let's update the CSV file by adding a new row.


In [9]:
# Modify the data to simulate an update
updated_data = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Charlie", "David"],
    "age": [23, 30, 35, 40]
})

# Update the dataset on GCS
datasets.update_dataset(updated_data, filename, data_format="csv")


Dataset 'people_data.csv' uploaded successfully.


The `update_dataset` method successfully overwrites the existing `people_data.csv` with the updated dataset. 

---

## 5. List All Datasets in GCS

We can list all datasets (files) in the GCS bucket. This is useful for checking which datasets are available.


In [10]:
# List all datasets in the bucket
datasets_list = datasets.list_datasets()

# Display the list of datasets
datasets_list


['delimited_json.json',
 'events_response_positions_updated.pkl',
 'financial_data_json.json',
 'line_delimited_json.json',
 'people_data.csv',
 'sample_data.csv',
 'table-1_data_new_line_delimited_json.json']

The `list_datasets` method returns a list of all files (datasets) in the GCS bucket. You can optionally filter them by a prefix (if you have a naming convention).

---

## 6. Delete a Dataset

If we no longer need a dataset, we can delete it using the `delete_dataset` method. Let's delete the `people_data.csv` file.


In [11]:
# Delete the dataset from GCS
dataset_deleted = datasets.delete_dataset(filename)

# Check if the dataset was successfully deleted
dataset_deleted


Dataset 'people_data.csv' deleted successfully.


True

The `delete_dataset` method successfully deletes the `people_data.csv` dataset from Google Cloud Storage.

---

## Conclusion

In this notebook, we've demonstrated the core functionalities of the `Datasets` class, including:
- Uploading datasets in various formats.
- Reading datasets from Google Cloud Storage.
- Updating and overwriting existing datasets.
- Listing available datasets.
- Deleting datasets from the bucket.

These functions provide an easy-to-use interface for managing datasets in Google Cloud Storage. You can adapt these methods to work with different types of data and file formats depending on your use case.

---

### Notes:
- Ensure you have set up Google Cloud authentication (via `GOOGLE_APPLICATION_CREDENTIALS`).
- The `datasets` object assumes the default project and bucket, but you can provide custom values if necessary.
